# Painel Financeiro para Pequena Empresa

DRE por competência, fluxo de caixa por regime de caixa, inadimplência por cliente, projeção de recebíveis e ponto de equilíbrio.

Este notebook gera a base, roda a análise e publica os dados numa planilha do Google. O painel em `index.html` lê essa planilha ao vivo, então basta rodar este notebook de novo sempre que quiser atualizar os números exibidos.

> ### ⚠️ Dados fictícios
> O escritório contábil não existe. Clientes, honorários e lançamentos são gerados por script, já que dados financeiros de empresas reais não podem ser publicados. A metodologia é real.

## 1. Gerar a base

In [ ]:
import random
from datetime import date, timedelta

import pandas as pd

SEED = 21
random.seed(SEED)

INICIO = date(2024, 1, 1)
FIM = date(2025, 12, 31)
N_CLIENTES = 58

PRIMEIROS = ["Alfa", "Bravo", "Cedro", "Delta", "Estrela", "Fluxo", "Girassol",
             "Horizonte", "Ipê", "Jangada", "Kappa", "Luminar", "Meridiano", "Norte",
             "Oliva", "Pampa", "Quero", "Raiz", "Serra", "Trevo", "União", "Vetor"]
SEGUNDOS = ["Comércio", "Serviços", "Alimentos", "Construções", "Logística",
            "Tecnologia", "Confecções", "Distribuidora", "Consultoria", "Indústria"]
SUFIXOS = ["Ltda", "ME", "EIRELI", "Ltda ME"]

SEGMENTOS = ["Varejo", "Serviços", "Indústria", "Construção Civil", "Saúde"]

# planos de honorário mensal do escritorio
PLANOS = [("Básico", 350.0, 0.40), ("Intermediário", 650.0, 0.38), ("Completo", 1100.0, 0.22)]


def gerar_clientes():
    clientes, usados = [], set()
    for i in range(N_CLIENTES):
        for _ in range(80):
            nome = f"{random.choice(PRIMEIROS)} {random.choice(SEGUNDOS)} {random.choice(SUFIXOS)}"
            if nome not in usados:
                break
        usados.add(nome)

        plano = random.choices(PLANOS, [p[2] for p in PLANOS])[0]
        # cada cliente tem um perfil de pagamento que se repete ao longo do tempo
        perfil = random.choices(
            ["pontual", "atraso_leve", "atraso_frequente", "inadimplente"],
            [0.55, 0.28, 0.12, 0.05]
        )[0]

        dias_entrada = random.randint(0, 500)
        clientes.append({
            "id_cliente": 3000 + i,
            "razao_social": nome,
            "segmento": random.choice(SEGMENTOS),
            "plano": plano[0],
            "honorario_mensal": plano[1],
            "data_inicio_contrato": INICIO + timedelta(days=dias_entrada),
            "perfil_pagamento": perfil,
        })
    return pd.DataFrame(clientes)


# media de dias de atraso e desvio, por perfil
ATRASO = {
    "pontual":            (0, 1),
    "atraso_leve":        (6, 4),
    "atraso_frequente":   (18, 9),
    "inadimplente":       (55, 25),
}
# chance de o mes ficar em aberto (nao pago ate o fim da base)
PROB_ABERTO = {"pontual": 0.01, "atraso_leve": 0.03, "atraso_frequente": 0.10, "inadimplente": 0.35}


def gerar_contas_receber(clientes):
    linhas, idc = [], 900000
    for _, c in clientes.iterrows():
        mes = date(c["data_inicio_contrato"].year, c["data_inicio_contrato"].month, 5)
        while mes <= FIM:
            if mes >= c["data_inicio_contrato"]:
                vencimento = mes
                media, desvio = ATRASO[c["perfil_pagamento"]]

                em_aberto = (mes > date(2025, 10, 1) and
                            random.random() < PROB_ABERTO[c["perfil_pagamento"]])

                if em_aberto:
                    pagamento = None
                    status = "Em aberto"
                else:
                    atraso = max(0, round(random.gauss(media, desvio)))
                    pagamento = vencimento + timedelta(days=atraso)
                    if pagamento > FIM:
                        pagamento = None
                        status = "Em aberto"
                    else:
                        status = "Pago"

                linhas.append({
                    "id_titulo": idc,
                    "id_cliente": c["id_cliente"],
                    "competencia": mes.strftime("%Y-%m"),
                    "data_vencimento": vencimento,
                    "data_pagamento": pagamento,
                    "valor": c["honorario_mensal"],
                    "status": status,
                })
                idc += 1

            # proximo mes
            if mes.month == 12:
                mes = date(mes.year + 1, 1, 5)
            else:
                mes = date(mes.year, mes.month + 1, 5)

    return pd.DataFrame(linhas)


# despesas fixas mensais do escritorio
DESPESAS_FIXAS = [
    ("Folha de pagamento", 12800.0, 0.02),
    ("Aluguel e condomínio", 2600.0, 0.00),
    ("Softwares e assinaturas", 780.0, 0.02),
    ("Contabilidade própria e taxas", 420.0, 0.01),
    ("Internet e telefonia", 260.0, 0.01),
    ("Marketing e comercial", 500.0, 0.15),
]

# despesas variaveis, como percentual da receita realizada do mes
DESPESAS_VARIAVEIS_PCT = 0.06  # comissao de indicacao e material de trabalho


def gerar_contas_pagar():
    linhas, idp = [], 700000
    mes = date(INICIO.year, INICIO.month, 1)
    valores = {d[0]: d[1] for d in DESPESAS_FIXAS}

    while mes <= FIM:
        for nome, valor_base, variacao in DESPESAS_FIXAS:
            valores[nome] *= (1 + random.uniform(-variacao, variacao * 1.6))
            linhas.append({
                "id_lancamento": idp,
                "categoria": nome,
                "competencia": mes.strftime("%Y-%m"),
                "data_vencimento": date(mes.year, mes.month, random.choice([5, 10, 15, 20])),
                "valor": round(valores[nome], 2),
                "tipo": "Fixa",
            })
            idp += 1

        if mes.month == 12:
            mes = date(mes.year + 1, 1, 1)
        else:
            mes = date(mes.year, mes.month + 1, 1)

    return pd.DataFrame(linhas)

In [ ]:
clientes = gerar_clientes()
receber = gerar_contas_receber(clientes)
pagar = gerar_contas_pagar()

clientes.to_csv("clientes.csv", index=False, encoding="utf-8")
receber.to_csv("contas_receber.csv", index=False, encoding="utf-8")
pagar.to_csv("contas_pagar.csv", index=False, encoding="utf-8")

print(f"clientes.csv         {len(clientes):>6} linhas")
print(f"contas_receber.csv   {len(receber):>6} linhas")
print(f"contas_pagar.csv     {len(pagar):>6} linhas")

## 2. Analisar

### DRE por competência
Receita e despesa contadas pelo mês a que se referem, não pela data em que o dinheiro mudou de mão. É o regime que mostra se o negócio é lucrativo.

In [ ]:
import pandas as pd

HOJE = pd.Timestamp("2025-12-31")


def carregar():
    cl = pd.read_csv("clientes.csv", parse_dates=["data_inicio_contrato"])
    rc = pd.read_csv("contas_receber.csv",
                     parse_dates=["data_vencimento", "data_pagamento"])
    pg = pd.read_csv("contas_pagar.csv", parse_dates=["data_vencimento"])
    return cl, rc, pg


# ---------------------------------------------------------------------
# DRE mensal
# ---------------------------------------------------------------------

def dre_mensal(rc, pg):
    """
    DRE por competência, ou seja, pelo mês a que a receita e a despesa
    se referem, não pela data em que o dinheiro efetivamente mudou de
    mão. É o regime que mostra se o negócio é lucrativo.
    """
    receita = rc.groupby("competencia")["valor"].sum().rename("receita")
    despesa = pg.groupby("competencia")["valor"].sum().rename("despesa")

    dre = pd.concat([receita, despesa], axis=1).fillna(0).reset_index()
    dre = dre.rename(columns={"index": "competencia"})
    dre["resultado"] = dre["receita"] - dre["despesa"]
    dre["margem_pct"] = (dre["resultado"] / dre["receita"] * 100).round(1)
    return dre.sort_values("competencia").reset_index(drop=True)

cl, rc, pg = carregar()
dre = dre_mensal(rc, pg)
dre.tail(6).round(1)

### Fluxo de caixa por regime de caixa
Pela data em que o pagamento efetivamente aconteceu. É o que explica por que o mês fecha no vermelho mesmo com contrato assinado: a receita da competência entrou atrasada, no mês seguinte.

In [ ]:
def caixa_mensal(rc, pg):
    """
    Fluxo de caixa por regime de caixa, pela data em que o pagamento
    aconteceu, não pela competência. É o que explica por que o mês
    fecha no vermelho mesmo com contrato assinado: o dinheiro daquele
    mês entrou atrasado, no mês seguinte.
    """
    pago = rc[rc["status"] == "Pago"].copy()
    pago["mes_recebimento"] = pago["data_pagamento"].dt.to_period("M").astype(str)
    entradas = pago.groupby("mes_recebimento")["valor"].sum().rename("entradas")

    pg = pg.copy()
    pg["mes_pagamento"] = pg["data_vencimento"].dt.to_period("M").astype(str)
    saidas = pg.groupby("mes_pagamento")["valor"].sum().rename("saidas")

    caixa = pd.concat([entradas, saidas], axis=1).fillna(0).reset_index()
    caixa = caixa.rename(columns={"index": "mes"})
    caixa["saldo_mes"] = caixa["entradas"] - caixa["saidas"]
    caixa["saldo_acumulado"] = caixa["saldo_mes"].cumsum()
    return caixa.sort_values("mes").reset_index(drop=True)

caixa = caixa_mensal(rc, pg)
caixa.tail(6).round(1)

### Inadimplência por cliente
Atraso médio de pagamento de cada cliente, sobre os títulos já pagos.

In [ ]:
def atraso_medio(rc):
    """Dias entre vencimento e pagamento, só para títulos já pagos."""
    pago = rc[rc["status"] == "Pago"].copy()
    pago["dias_atraso"] = (pago["data_pagamento"] - pago["data_vencimento"]).dt.days
    return pago


def inadimplencia_por_cliente(rc, clientes):
    pago = atraso_medio(rc)

    r = pago.groupby("id_cliente").agg(
        titulos_pagos=("id_titulo", "count"),
        atraso_medio=("dias_atraso", "mean"),
        atraso_maximo=("dias_atraso", "max"),
        pontual_pct=("dias_atraso", lambda s: (s <= 5).mean() * 100),
    ).reset_index()

    aberto = rc[rc["status"] == "Em aberto"].groupby("id_cliente").agg(
        titulos_em_aberto=("id_titulo", "count"),
        valor_em_aberto=("valor", "sum"),
    ).reset_index()

    r = r.merge(aberto, on="id_cliente", how="left").fillna({
        "titulos_em_aberto": 0, "valor_em_aberto": 0})
    r = r.merge(clientes[["id_cliente", "razao_social", "segmento",
                          "plano", "honorario_mensal"]], on="id_cliente")

    r["risco"] = pd.cut(
        r["atraso_medio"], [-1, 5, 15, 30, 9999],
        labels=["Pontual", "Atenção", "Risco", "Alto risco"])

    return r.sort_values("atraso_medio", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------------
# Projeção de recebíveis
# ---------------------------------------------------------------------

def projetar_recebimento(rc, inadimplencia, dias=90):
    """
    Estima quando os títulos em aberto vão entrar, aplicando a cada
    cliente o atraso médio dele mesmo, e não uma média geral. É o que
    transforma a data de vencimento contratual na data provável de
    entrada no caixa.
    """
    aberto = rc[rc["status"] == "Em aberto"].merge(
        inadimplencia[["id_cliente", "atraso_medio"]], on="id_cliente", how="left")
    aberto["atraso_medio"] = aberto["atraso_medio"].fillna(
        inadimplencia["atraso_medio"].median())

    aberto["previsao_recebimento"] = aberto["data_vencimento"] + \
        pd.to_timedelta(aberto["atraso_medio"].round(), unit="D")

    horizonte = HOJE + pd.Timedelta(days=dias)
    janela = aberto[aberto["previsao_recebimento"] <= horizonte].copy()
    janela["semana"] = janela["previsao_recebimento"].dt.to_period("W").astype(str)

    faixas = [30, 60, 90]
    resumo = {}
    for f in faixas:
        limite = HOJE + pd.Timedelta(days=f)
        resumo[f"ate_{f}_dias"] = round(
            aberto[aberto["previsao_recebimento"] <= limite]["valor"].sum(), 2)

    return aberto.sort_values("previsao_recebimento"), resumo

inad = inadimplencia_por_cliente(rc, cl)
inad["risco"].value_counts()

### Projeção de recebíveis
Aplica a cada cliente o atraso médio dele mesmo, não uma média geral, para transformar a data de vencimento contratual na data provável de entrada no caixa.

In [ ]:
def projetar_recebimento(rc, inadimplencia, dias=90):
    """
    Estima quando os títulos em aberto vão entrar, aplicando a cada
    cliente o atraso médio dele mesmo, e não uma média geral. É o que
    transforma a data de vencimento contratual na data provável de
    entrada no caixa.
    """
    aberto = rc[rc["status"] == "Em aberto"].merge(
        inadimplencia[["id_cliente", "atraso_medio"]], on="id_cliente", how="left")
    aberto["atraso_medio"] = aberto["atraso_medio"].fillna(
        inadimplencia["atraso_medio"].median())

    aberto["previsao_recebimento"] = aberto["data_vencimento"] + \
        pd.to_timedelta(aberto["atraso_medio"].round(), unit="D")

    horizonte = HOJE + pd.Timedelta(days=dias)
    janela = aberto[aberto["previsao_recebimento"] <= horizonte].copy()
    janela["semana"] = janela["previsao_recebimento"].dt.to_period("W").astype(str)

    faixas = [30, 60, 90]
    resumo = {}
    for f in faixas:
        limite = HOJE + pd.Timedelta(days=f)
        resumo[f"ate_{f}_dias"] = round(
            aberto[aberto["previsao_recebimento"] <= limite]["valor"].sum(), 2)

    return aberto.sort_values("previsao_recebimento"), resumo


# ---------------------------------------------------------------------
# Ponto de equilíbrio
# ---------------------------------------------------------------------

def ponto_de_equilibrio(dre, despesas_variaveis_pct=0.06):
    """
    Faturamento necessário para cobrir o custo fixo, dado que parte da
    despesa cresce junto com a receita.

    O custo fixo é isolado subtraindo da despesa média a parcela que
    varia com o faturamento. Ponto de equilíbrio = custo fixo dividido
    por um menos o percentual variável, porque cada real faturado
    também gera despesa variável e só a diferença cobre o fixo.

    O percentual variável não é calculado dos dados, é assumido pela
    premissa de geração da base, e num caso real precisa ser levantado
    com o cliente a partir da composição real dos custos.
    """
    receita_media = dre["receita"].mean()
    despesa_media = dre["despesa"].mean()

    custo_variavel_medio = receita_media * despesas_variaveis_pct
    custo_fixo_medio = despesa_media - custo_variavel_medio

    equilibrio = custo_fixo_medio / (1 - despesas_variaveis_pct)

    return {
        "receita_media_mensal": round(receita_media, 2),
        "despesa_media_mensal": round(despesa_media, 2),
        "custo_fixo_estimado": round(custo_fixo_medio, 2),
        "custo_variavel_estimado": round(custo_variavel_medio, 2),
        "ponto_de_equilibrio": round(equilibrio, 2),
        "folga_sobre_equilibrio_pct": round(receita_media / equilibrio * 100 - 100, 1),
    }

projecao, resumo_proj = projetar_recebimento(rc, inad)
resumo_proj

### Ponto de equilíbrio
Faturamento necessário para cobrir o custo fixo, isolando a parcela da despesa que varia com a receita.

In [ ]:
pe = ponto_de_equilibrio(dre)
pe

## 3. Publicar no Google Sheets

Escreve as tabelas na planilha que alimenta o painel. Pede autorização do Google na primeira execução.

In [ ]:
def _autenticar():
    try:
        from google.colab import auth
        from google.auth import default
        import gspread
    except ImportError as erro:
        raise RuntimeError(
            "Este módulo foi feito para rodar no Google Colab."
        ) from erro

    auth.authenticate_user()
    credenciais, _ = default()
    return gspread.authorize(credenciais)

In [ ]:
def _preparar(df):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_period_dtype(df[c]):
            df[c] = df[c].astype(str)
        elif df[c].dtype.name in ("category", "object"):
            df[c] = df[c].astype(str)
    return df.where(pd.notna(df), "")


def _gravar_aba(planilha, nome, df):
    df = _preparar(df)
    try:
        aba = planilha.worksheet(nome)
        aba.clear()
    except Exception:
        aba = planilha.add_worksheet(
            title=nome, rows=max(len(df) + 10, 100), cols=max(len(df.columns) + 2, 20))
    aba.update(values=[list(df.columns)] + df.values.tolist(), range_name="A1")
    aba.freeze(rows=1)
    return len(df)

In [ ]:
def exportar(nome_planilha="Escritório Contábil - Painel Financeiro"):
    cl, rc, pg = carregar()
    dre = dre_mensal(rc, pg)
    caixa = caixa_mensal(rc, pg)
    inad = inadimplencia_por_cliente(rc, cl)
    projecao, resumo_proj = projetar_recebimento(rc, inad)
    pe = ponto_de_equilibrio(dre)

    # titulos individuais, base do painel para filtro por cliente e mes
    titulos = rc.merge(
        cl[["id_cliente", "razao_social", "segmento", "plano"]],
        on="id_cliente", how="left")
    titulos = titulos.merge(inad[["id_cliente", "risco"]], on="id_cliente", how="left")

    conexao = _autenticar()
    try:
        planilha = conexao.open(nome_planilha)
    except Exception:
        planilha = conexao.create(nome_planilha)

    abas = {
        "dre_mensal": dre,
        "caixa_mensal": caixa,
        "inadimplencia": inad,
        "titulos": titulos,
        "contas_pagar": pg,
        "projecao_recebiveis": projecao,
    }
    for nome, dados in abas.items():
        linhas = _gravar_aba(planilha, nome, dados)
        print(f"{nome:<20} {linhas:>6} linhas")

    # aba de indicadores unicos, formato chave-valor
    pe_df = pd.DataFrame([{"indicador": k, "valor": v} for k, v in pe.items()])
    for k, v in resumo_proj.items():
        pe_df.loc[len(pe_df)] = [k, v]
    _gravar_aba(planilha, "indicadores", pe_df)

    try:
        planilha.del_worksheet(planilha.worksheet("Sheet1"))
    except Exception:
        pass

    print()
    print(f"Planilha disponível em {planilha.url}")
    return planilha.url

url = exportar()

## 4. Conectar o painel

Copie o link acima. No Sheets: Arquivo, Compartilhar, Publicar na web, escolha a aba `titulos` em formato CSV, copie o link, repita para a aba `contas_pagar`. Cole os dois endereços nas constantes `CSV_TITULOS` e `CSV_PAGAR` no início do `index.html`.